# 🏔️ TrailSense Phase 11 — Hindi/Marathi LLM Fine-Tuning
> Fine-tune Gemma 2B on navigation & safety Q&A pairs in Hindi and Marathi using QLoRA/Unsloth on Google Colab T4 GPU.

**Steps:** Environment → GPU Check → Load Gemma 2B → Load Dataset → Fine-Tune → Evaluate → Export GGUF

## Step 1: Install Dependencies (run once, ~3-5 min)

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets sentencepiece protobuf
print("✅ Dependencies installed.")


## Step 2: Verify GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ CUDA not available! Switch to GPU runtime in Colab: Runtime > Change runtime type > T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}")
print(f"✅ VRAM: {vram_gb:.1f} GB")
print(f"✅ CUDA version: {torch.version.cuda}")


## Step 3: HuggingFace Login
> Paste your HuggingFace token below. Get it from: https://huggingface.co/settings/tokens
> Also accept Gemma's license at: https://huggingface.co/google/gemma-2b

In [ ]:
from huggingface_hub import login
from getpass import getpass

token = getpass("Enter your HuggingFace token (hf_...): ")
login(token=token)
print("✅ Logged in to HuggingFace!")


## Step 4: Load Meta Llama 3.2 1B Instruct (4-bit QLoRA-ready)
> Using Unsloth's optimized Llama 3.2 1B — fast mobile-friendly fine-tuning with 128k Indic token vocabulary.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Suitable for navigation Q&A
dtype = None           # Auto-detect (bfloat16 on Ampere+, float16 on older)
load_in_4bit = True    # 4-bit QLoRA — fits in T4 16GB VRAM comfortably

print("Loading Llama 3.2 1B-Instruct (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Llama 3.2 1B-Instruct loaded!")
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")


## Step 5: Apply QLoRA Adapters
> We only train ~0.5% of parameters — LoRA rank 16 on attention & MLP layers.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% more efficient
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied!")
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## Step 6: Load Hindi/Marathi Navigation Dataset
> 100 instruction-tuning pairs covering shelter, water, exit, and emergency navigation in Hindi and Marathi.

In [ ]:
import json
import urllib.request
from datasets import Dataset

# Load dataset — paste your phase11_dataset.json content here OR upload the file to Colab
# Option A: Upload phase11_dataset.json to Colab (Files panel on left) and load it:
try:
    with open("phase11_dataset.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print(f"✅ Dataset loaded from file: {len(raw_data)} examples")
except FileNotFoundError:
    print("❌ phase11_dataset.json not found!")
    print("Please upload phase11_dataset.json to Colab using the Files panel (folder icon on left).")
    raise

# Preview first example
print("\n--- Sample example ---")
print(f"Instruction: {raw_data[0]['instruction']}")
print(f"Input: {raw_data[0]['input'][:100]}...")
print(f"Output: {raw_data[0]['output'][:100]}...")


## Step 7: Format Dataset for Instruction Fine-Tuning

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma",
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        messages = [
            {"role": "user", "content": f"{inst}\n\nContext:\n{inp}"},
            {"role": "assistant", "content": out}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
    return {"text": texts}

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset formatted: {len(hf_dataset)} examples")
print(f"\nSample formatted prompt (first 300 chars):")
print(hf_dataset[0]["text"][:300])


## Step 8: Evaluate BEFORE Fine-Tuning
> Run baseline test on Hindi and Marathi questions. Save these answers for comparison.

In [ ]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)
test_questions = [
    {"lang": "Hindi", "instruction": "मुझे सबसे नजदीकी पानी का स्रोत बताओ।", "input": "वर्तमान स्थिति: पर्वत पगडंडी। निकटतम आश्रय: शिखर आश्रय, 320m। निकटतम जल स्रोत: पर्वत झरना, 580m। निकटतम निकास: पूर्वी द्वार, 2.1km。"},
    {"lang": "Hindi", "instruction": "मैं खो गया हूं, मुझे क्या करना चाहिए?", "input": "वर्तमान स्थिति: घना जंगल। निकटतम आश्रय: वन शरण, 400m। निकटतम जल स्रोत: वन नाला, 600m। निकटतम निकास: मुख्य मार्ग, 3km。"},
    {"lang": "Hindi", "instruction": "आपातकाल में मुझे क्या करना चाहिए?", "input": "वर्तमान स्थिति: खुला पर्वत। निकटतम आश्रय: पर्वत केबिन, 200m। निकटतम जल स्रोत: झरना, 800m। निकटतम निकास: घाटी रास्ता, 4km。"},
    {"lang": "Marathi", "instruction": "सर्वात जवळचा निवारा कुठे आहे?", "input": "सध्याची स्थिती: पर्वत मार्ग। निकटतम निवारा: शिखर निवारा, 280m। निकटतम जलस्त्रोत: पर्वत झरा, 520m। निकटतम बाहेर: पूर्व द्वार, 1.9km。"},
    {"lang": "Marathi", "instruction": "मी हरवलो आहे, मला काय करावे?", "input": "सध्याची स्थिती: घनदाट जंगल। निकटतम निवारा: वन निवारा, 400m। निकटतम जलस्त्रोत: नाला, 500m। निकटतम बाहेर: मुख्य रस्ता, 2.8km。"}
]
before_answers = []
print("=" * 60)
print("BEFORE FINE-TUNING — Baseline Answers")
print("=" * 60)

for q in test_questions:
    prompt = f"Below is an instruction that describes a trail navigation task. Write a response in the same language as the instruction.\n\n### Instruction:\n{q['instruction']}\n\n### Input:\n{q['input']}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = full_text.split("### Response:\n")[-1].strip() if "### Response:\n" in full_text else full_text
    before_answers.append(answer)
    print(f"\n[{q['lang']}] Q: {q['instruction']}\nA: {answer}\n" + "-"*40)

print("\n✅ Baseline answers saved for comparison.")


## Step 9: Fine-Tune with QLoRA (SFT)
> Training takes ~30-60 minutes on Colab T4. Watch the loss decrease.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)  # Switch back to training mode

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=5,             # 5 epochs for optimal convergence
        learning_rate=1e-4,             # Stable learning rate for Gemma 2B
        max_grad_norm=1.0,              # Gradient clipping to prevent loss explosion
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="trailsense_lora_output",
        report_to="none",
    ),
)

print("🚀 Starting fine-tuning...")
trainer_stats = trainer.train()

print("\n✅ Fine-tuning complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")


## Step 10: Evaluate AFTER Fine-Tuning
> Compare answers with baseline. The fine-tuned model should give more fluent and accurate Hindi/Marathi answers.

In [ ]:
FastLanguageModel.for_inference(model)

after_answers = []
print("=" * 60)
print("AFTER FINE-TUNING — Improved Answers")
print("=" * 60)

for i, q in enumerate(test_questions):
    prompt = f"Below is an instruction that describes a trail navigation task. Write a response in the same language as the instruction.\n\n### Instruction:\n{q['instruction']}\n\n### Input:\n{q['input']}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    answer = full_text.split("### Response:\n")[-1].strip() if "### Response:\n" in full_text else full_text
    after_answers.append(answer)
    print(f"\n[{q['lang']}] Q: {q['instruction']}")
    print(f"BEFORE: {before_answers[i][:150]}")
    print(f"AFTER:  {answer[:150]}")
    print("-" * 40)

print("\n✅ Comparison complete. Review the before/after answers above.")
print("If AFTER answers are more fluent and accurate, proceed to export.")
print("If not, consider more training epochs or a larger dataset.")


## Step 11: Save LoRA Adapter
> Save just the LoRA weights (small, ~100MB) before merging.

In [ ]:
# Save LoRA adapter weights only
model.save_pretrained("trailsense_lora_adapter")
tokenizer.save_pretrained("trailsense_lora_adapter")
print("✅ LoRA adapter saved to: trailsense_lora_adapter/")


## Step 12: Merge & Export to GGUF (4-bit Q4_K_M)
> Merge LoRA into base model and export to GGUF format for Android deployment.
> This creates a ~1.5GB quantized model file.

In [ ]:
# Merge LoRA adapters into base model and export GGUF
print("Merging LoRA and exporting to GGUF Q4_K_M...")
print("This may take 5-10 minutes...")

model.save_pretrained_gguf(
    "trailsense_llama3.2_hindi_marathi",
    tokenizer,
    quantization_method="q4_k_m"  # 4-bit, good balance of size/quality
)

import os
gguf_files = [f for f in os.listdir(".") if f.endswith(".gguf")]
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"✅ GGUF exported: {f} ({size_mb:.0f} MB)")


## Step 13: Download GGUF Model
> Download the fine-tuned GGUF model file to your local machine.

In [ ]:
from google.colab import files
import glob

gguf_files = glob.glob("*.gguf")
if gguf_files:
    gguf_path = gguf_files[0]
    size_mb = os.path.getsize(gguf_path) / 1e6
    print(f"Downloading: {gguf_path} ({size_mb:.0f} MB)...")
    files.download(gguf_path)
    print("✅ Download started!")
else:
    print("❌ No GGUF file found. Run Step 12 first.")


## ✅ Phase 11 Complete!

### What to do next:
1. **Review** the before/after answers (Step 10) — approve only if quality improved
2. **Download** the `trailsense_hindi_marathi-Q4_K_M.gguf` file (Step 13)
3. **Share the GGUF file path** with your developer — they will integrate it into the Android app

### Android Integration (Phase 11b):
- Replace the existing MediaPipe model with the fine-tuned GGUF
- Switch `LlmAssistant.java` to use llama.cpp/MLC-LLM for GGUF inference
- Re-run Phase 7 offline verification in Hindi and Marathi

> **Note:** If the fine-tuned model does NOT show genuine improvement, do NOT replace the English model.